# Command(resume=)으로 실행 재개

잘못된 승인 입력을 다시 요청하고 올바른 값으로 같은 스레드를 완료하는 과정을 확인합니다.

## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.

In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11

Note: you may need to restart the kernel to use updated packages.


## 입력 검증 노드 준비

`interrupt()`를 반복문 안에 두고 허용할 문자열을 제한합니다.

In [2]:
# approve와 reject만 허용하는 interrupt 입력 검증 노드를 준비합니다.
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

class State(TypedDict):
    request: str
    decision: str

def review(state: State) -> dict:
    prompt = {"request": state["request"], "allowed": ["approve", "reject"]}
    while True:
        decision = interrupt(prompt)
        if decision in {"approve", "reject"}:
            return {"decision": decision}
        prompt = {**prompt, "error": f"허용하지 않는 값: {decision}"}

builder = StateGraph(State)
builder.add_node("review", review)
builder.add_edge(START, "review")
builder.add_edge("review", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "review-2402"}}

## 처음 중단하고 잘못된 값 전달

각 호출 뒤 `__interrupt__`에 담긴 질문을 확인합니다.

In [3]:
# 첫 중단 뒤 maybe를 전달해 오류가 추가된 두 번째 interrupt 값을 읽습니다.
first = graph.invoke({"request": "publish-local", "decision": ""}, config)
second = graph.invoke(Command(resume="maybe"), config)
print("first prompt:", first["__interrupt__"][0].value)
print("second prompt:", second["__interrupt__"][0].value)
assert "error" in second["__interrupt__"][0].value

first prompt: {'request': 'publish-local', 'allowed': ['approve', 'reject']}
second prompt: {'request': 'publish-local', 'allowed': ['approve', 'reject'], 'error': '허용하지 않는 값: maybe'}


## 올바른 값으로 완료

같은 `thread_id`에 `Command(resume=...)`을 전달합니다.

In [4]:
# 같은 thread_id에 approve를 전달해 decision과 종료 상태를 확인합니다.
final = graph.invoke(Command(resume="approve"), config)
snapshot = graph.get_state(config)
print("final:", final)
print("next:", snapshot.next)
print("history count:", len(list(graph.get_state_history(config))))
assert final["decision"] == "approve" and snapshot.next == ()

final: {'request': 'publish-local', 'decision': 'approve'}
next: ()
history count: 3
